# Build a first RAG baseline

This notebook is a coding guide. Its cells are Markdown only; add your own code cells beneath each step. The aim is to build the simplest complete RAG using the processed chunks already in this repository:

`data/processed/chunks.jsonl` → embeddings → local vector store → retrieval → answer grounded in retrieved text, with paper/page citations.

Then use the ten cross-paper questions in `docs/rag-build-log.md` as a small evaluation set. Four papers are enough to build and exercise a prototype. They may not contain enough evidence to answer every question, so an honest 'the papers do not establish this' is a valid result.

Keep this first version deliberately simple: one user question, one retrieval step, one answer step. You do not need paper summaries, per-paper question generation, agents, or LangGraph orchestration to get the baseline working.

In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, model_validator
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from tqdm import tqdm
from langchain_core.documents import Document

load_dotenv()

CHUNKS_PATH = Path("../data/processed/chunks.jsonl")
SCREENED_OUTPUT_PATH = Path("../data/processed/screened_chunks_v2_prompt.jsonl")
REVIEWED_OUTPUT_PATH = Path("../data/processed/reviewed_screened_chunks_v2_prompt.jsonl")

MODEL_NAME = "gpt-6-luna"
MODEL_PROVIDER = "openai"
MODEL_MAX_RETRIES = 3
MODEL_TIMEOUT_SECONDS = 15.0
MODEL_TEMPERATURE = 1
SAMPLE_SIZE = 25
RANDOM_SEED = 56


def save_jsonl(records, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open(mode="w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record) + "\n")


def load_jsonl(path):
    path = Path(path)
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

## 1. Confirm the input corpus

Use the existing ingestion output at `data/processed/chunks.jsonl`; do not reload and reprocess the PDFs for this notebook. The corpus currently has 162 chunks from four papers. Each record contains the chunk text and metadata such as filename, document ID, PDF page, and chunk ID.

In your first code cell, load a few JSONL records and inspect their shape. Confirm that the text is non-empty and the provenance fields you need are present. Count documents and chunks so you know what entered the index.

If you spot a missing title, use the filename as a citation label for now. Preserve the original metadata rather than flattening it away: source and page information will be needed when the model cites evidence. Note extraction issues, especially flattened tables/equations, but do not pause the baseline to fix every PDF artifact.

In [2]:
chunks_dict = load_jsonl(CHUNKS_PATH)
print(f"Number of chunks: {len(chunks_dict)}")
print(f"Number of chunks with no text: {len([chunk for chunk in chunks_dict if not chunk['page_content']])}")
print(f"Keys in a chunk: {chunks_dict[0].keys()}")
print(f"Keys in a chunk: {chunks_dict[0]['metadata']}")

Number of chunks: 162
Number of chunks with no text: 0
Keys in a chunk: dict_keys(['metadata', 'page_content'])
Keys in a chunk: {'chunk_id': 'a42ab7c4d89ca0a595ab4488c354966a0a1f6da7393016f906d716cdcbc5b115', 'chunk_index': 0, 'chunk_text_hash': '4b5ff232a78c7191daf599977d6135e5d393c588844b6999eaa4e5709b707d57', 'creationdate': '2026-09-10T16:17:00+01:00', 'creator': 'PyPDF', 'document_hash': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109', 'document_id': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109', 'filename': '2024156785.pdf', 'page': 0, 'page_label': '1', 'producer': 'OpenPDF 1.4.2', 'source': 'data/raw/2024156785.pdf', 'total_pages': 7}


## 2. Turn records into searchable documents

Create one LangChain `Document` per chunk (or use the document type expected by your chosen vector-store integration). Put the chunk's `page_content` in the document text and retain its metadata. At minimum, keep:

- `document_id` and `filename` (or a readable paper title);
- `page` and `page_label` when available;
- `chunk_id`;
- `source` or another path/identifier for the original paper.

Inspect one converted document and check that metadata survives. A common bug is embedding only the text and then discovering that the retrieved result cannot be traced back to a paper or page.

In [3]:
from langchain_core.documents import Document

chunks = []

for record in chunks_dict:
    chunk = Document(
        page_content=record['page_content'],
        metadata=record['metadata'])
    chunks.append(chunk)

print(len(chunks))

162


In [4]:
chunks[9].metadata

{'chunk_id': '2fb96f9ef3ae6459b540775fb79c9c7bafa4120d5cd1f116a6740b7e4f1bb687',
 'chunk_index': 9,
 'chunk_text_hash': '79d07badb983923edd9bc566372de4551664848547c5a14197edc85bb495fff7',
 'creationdate': '2026-09-10T16:17:00+01:00',
 'creator': 'PyPDF',
 'document_hash': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
 'document_id': '39ea32cd1e02652bbf1360a1b87103d0591d8abcbac8b6b3324257acbdc4e109',
 'filename': '2024156785.pdf',
 'page': 3,
 'page_label': '4',
 'producer': 'OpenPDF 1.4.2',
 'source': 'data/raw/2024156785.pdf',
 'total_pages': 7}

## 3. Choose an embedding model and create the local index

Use one embedding model consistently for indexing and search. This project already declares `langchain-openai`; if you use OpenAI embeddings, load the API key from your environment or a local `.env` file and never put the key in a notebook cell or commit it. The first embedding call may send the paper chunks to the selected provider, so use the provider you intend for this project.

Choose a persistent local vector store and save it under `data/vectorstore/`. The index should be rebuildable from `chunks.jsonl`; treat it as generated data, not the authoritative corpus. The project log says the vector-store directory is ignored by Git.

The current dependencies do not yet declare a specific persistent vector-store integration. Pick one LangChain-supported local store and add its package if needed (for example, the Chroma integration). Avoid comparing several stores at this stage. Create the index from the documents, then confirm that it exists and can be loaded again in a fresh notebook session.

Record the embedding model and vector-store type in a Markdown note or run record. Changing the embedding model later requires rebuilding the index with that same model.

Below we use `OpenAIEmbeddings` as embedding model and then use the langchain chroma integration to ingest the chunks in the chroma db with the collection name `football_match_predictions`

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize the OpenAI embedding model
embedding_model = OpenAIEmbeddings(
    # api_key=api_key,  # Not needed because we used load_dotenv
    model='text-embedding-3-small')

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../data/vectorstore/",
    collection_name="football_match_predictions" 
)

get two elements from the vector store

In [6]:
vector_store.get(
    limit=2,
    include=["documents", "metadatas", "embeddings"] )#['documents']

{'ids': ['7cea1dbb-bc4e-4be4-abe1-0cd4e012a416',
  '2774b1c4-80eb-41c3-b5ec-aea373c0d27b'],
 'embeddings': array([[-0.02107239,  0.00130272,  0.01799011, ..., -0.00885773,
          0.01039124,  0.00786591],
        [ 0.01855469,  0.03071594, -0.00163937, ...,  0.01163483,
          0.00462341,  0.00706863]], shape=(2, 1536)),
 'documents': ["Predicting Football Match Outcomes Using Event Data and Machine Learning\nAlgorithms\nHassard, P., & Kerr, D. (2024). Predicting Football Match Outcomes Using Event Data and Machine Learning\nAlgorithms. In H. Zheng, I. Cleland, A. Moore, H. Wang, D. Glass, J. Rafferty, R. Bond, & J. Wallace (Eds.),\nProceedings of the 35th Irish Systems and Signals Conference, ISSC 2024 (Proceedings of the 35th Irish\nSystems and Signals Conference, ISSC 2024). Institute of Electrical and Electronics Engineers Inc..\nhttps://doi.org/10.1109/ISSC61953.2024.10603147\nLink to publication record in Ulster University Research Portal\nPublished in:\nProceedings of the 

In [7]:
# Use the hidden native client inside your LangChain object
client = vector_store._client

# List all collections in the database
collections = client.list_collections()

print("Available collections:")
for col in collections:
    print(f"- {col.name}")


Available collections:
- football_match_predictions


## 4. Implement and inspect retrieval

Create a retriever that accepts a question and returns a small set of the most relevant chunks. Start with a straightforward similarity search and a modest `k` (for example, 4–6); tune it only after seeing results. Do not add hybrid search, query rewriting, or a reranker yet.

Try a few questions from the ten gold questions. Print each result with its rank, filename/title, page, and a readable excerpt. Check two things separately:

1. Is the passage actually relevant to the question?
2. Do results cover more than one paper when the question asks for cross-paper evidence?

Several retrieved chunks may come from the same paper. That can be correct, but it is not evidence of agreement across studies. Keep the document IDs visible while inspecting results.

In [8]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

question = "what are the best predicting features that can be constructed prematch"
results = retriever.invoke(question)

In [9]:
for rank, result in enumerate(results):
    print(rank)
    print(result.page_content)
    print("\n", "*" * 60, "\n")

0
of each match [11], so that the model can be used to
make prediction before the football match take place.
From the collected data, we have combined the rele-
vant features for both the home team and away team
by taking the difference between the values. For in-
stance, we combined the Overall Rating of the home
team and the Overall Rating of the away team by tak-
ing the difference between them. In other words, the
gap of the Overall Rating between the home team and
the away team is the feature that will be used in our
model building. Table 2 shows all the features that
were obtained after feature engineering.
In addition to the 26 features shown in Table 2, the
4 features shown in Table 3 were also included in our
dataset.
Since a team’s performance in the previous matches
is a good indicator of how well they would perform
in the next match. We have engineered our features
by taking the average value of 5 matches prior to the
current match. This means that the first 5 matches of
th

Maybe for later

## 5. Add grounded answer generation

Pass the question and retrieved passages to a chat model. The instruction should make the model use the supplied passages as evidence, distinguish what a paper reports from its own general knowledge, preserve important qualifiers, and say when the evidence is insufficient or studies cannot be compared.

Ask for citations tied to the retrieved metadata. A useful citation should identify the paper and page, not just say 'according to the context'. If two chunks have the same filename, include page numbers so the claim can be checked.

A basic answer prompt should tell the model to:

- answer only to the extent supported by the supplied passages;
- attach a paper/page citation to each key claim;
- distinguish a feature being mentioned or used from evidence that it improved prediction;
- avoid declaring one model or feature universally best from incomparable studies;
- state plainly when the retrieved evidence does not answer the question.

Keep the retrieved documents alongside the answer in the return value. Do not return only generated text; you will need the evidence to inspect citations and diagnose failures.

## 6. Run one end-to-end question

Connect the pieces in a small function or sequence:

1. accept a question;
2. retrieve relevant chunks;
3. format the text and source metadata as context;
4. generate a grounded answer;
5. return the answer together with retrieved sources.

Start with: **Which features appear consistently useful across the studies?** Inspect the retrieved evidence and answer manually. Notice whether the system distinguishes repeated feature use from consistent evidence of predictive usefulness. If the answer overclaims, check whether the fault is in retrieval, the prompt, or the evidence actually available in the four papers.

## 7. Use the ten gold questions as a lightweight evaluation set

Run all ten questions from `docs/rag-build-log.md` through the same end-to-end flow. You do not need to write a perfect reference answer for every question before starting. For each question, save the question, retrieved document/page/chunk IDs, answer, and a short review. A small JSONL file or a Markdown table is enough for the first pass.

For the review, record:

- **Corpus coverage:** answerable, partially answerable, or unsupported by these four papers.
- **Retrieval:** did the returned chunks contain relevant evidence, and were relevant papers missing?
- **Answer:** are the claims supported, appropriately qualified, and complete enough?
- **Citations:** do the cited paper and page actually support the attached claim?

This distinction matters: if none of the four papers discusses a question, retrieval cannot fetch evidence that is not in the corpus. Mark that as a coverage gap. If relevant evidence exists in the chunks but retrieval misses it, that is a retrieval problem. If retrieval finds good evidence but the response misstates it, that is a generation problem.

For model-comparison questions, studies may use different leagues, targets, data splits, and metrics. A careful answer may explain why the results do not support a fair overall ranking. That is a useful answer, not a failed eval.

## 8. Make one improvement at a time

After the first run, inspect the failed questions and choose the likely cause before changing anything:

- Relevant evidence exists but did not appear: try a different `k`, chunking change, metadata-aware retrieval, or later hybrid search.
- Evidence appeared but was buried among weak matches: consider a reranker after measuring the baseline.
- Evidence appeared but the answer overclaimed or lacked citations: improve the answer instructions or context formatting.
- The papers do not contain the evidence: mark the coverage gap. Decide separately whether to add papers; do not tune retrieval to hide a corpus limitation.

Change one thing, rerun the same ten questions, and compare with the saved first run. That gives you a meaningful check that the change helped. Synthetic passage-level questions can be added later if they help isolate retrieval quality, but they are optional.

## 9. Move stable code out of the notebook when ready

Once the basic flow works, move reusable pieces into the existing package under `src/rag_agent/` (for example, embedding/index creation, retrieval, and answer generation). Keep this notebook as the readable experiment and import the reusable functions into it. For the first pass, favor a small working vertical slice over a broad framework.

Do not add LangGraph just because it is in the project ecosystem. A single retrieval-and-answer path is enough for the baseline; orchestration becomes useful later if you add branching, retries, routing, or multi-step behavior.